In [1]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("/Users/ishika/Documents/AIProject/data/raw")

csv_files = sorted(RAW_DIR.glob("*.csv"))

print(f"CSV files found: {len(csv_files)}")

for file in csv_files:
    print(file.name)

CSV files found: 8
CheckingsOrSavings.csv
CreditCard2025.csv
MoneyTransfer.csv
Mortgage.csv
debtcollection31may.csv
debtcollectiondecember.csv
debtcollectionmay.csv
debtcollectionseptember.csv


In [2]:
# Step 2 — Our first notebook
file_summary = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)

    file_summary.append({
        "file": file.name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_complaint_ids": (
            df["Complaint ID"].duplicated().sum()
            if "Complaint ID" in df.columns
            else None
        )
    })

summary_df = pd.DataFrame(file_summary)
summary_df

,file,rows,columns,duplicate_complaint_ids
0,CheckingsOrSavings.csv,84194,16,0
1,CreditCard2025.csv,89979,16,0
2,MoneyTransfer.csv,84619,16,0
3,Mortgage.csv,24697,16,0
4,debtcollection31may.csv,81535,16,0
5,debtcollectiondecember.csv,80536,16,0
6,debtcollectionmay.csv,21959,16,0
7,debtcollectionseptember.csv,99175,16,0


In [3]:
summary_df["rows"].sum()

np.int64(566694)

In [4]:
# Step 3 — Validate the schema
sample_df = pd.read_csv(csv_files[0], low_memory=False)

sample_df.columns.tolist()

['Date received',
 'Product',
 'Sub-product',
 'Issue',
 'Sub-issue',
 'Consumer complaint narrative',
 'Company public response',
 'Company',
 'State',
 'ZIP code',
 'Tags',
 'Submitted via',
 'Date sent to company',
 'Company response to consumer',
 'Timely response?',
 'Complaint ID']

In [5]:
sample_df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2025-01-02T14:26:35.000Z,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,NaN,NaN,JPMORGAN CHASE & CO.,TN,37066,NaN,Phone,2025-01-02T14:41:19.000Z,Closed with explanation,Yes,11354262
1,2025-01-01T02:34:28.000Z,Checking or savings account,Checking account,Managing an account,Funds not handled or disbursed as instructed,Are deposits made to Bank of America actually ...,Company has responded to the consumer and the ...,"BANK OF AMERICA, NATIONAL ASSOCIATION",TN,37217,NaN,Web,2025-01-01T02:50:13.000Z,Closed with explanation,Yes,11345823
2,2025-01-01T00:54:23.000Z,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,NaN,NaN,JPMORGAN CHASE & CO.,SC,29306,Servicemember,Web,2025-01-01T01:17:26.000Z,Closed with explanation,Yes,11334911
3,2025-01-01T02:20:18.000Z,Checking or savings account,Savings account,Opening an account,Account opened without my consent or knowledge,NaN,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,NY,10037,NaN,Web,2025-01-01T02:53:35.000Z,Closed with explanation,Yes,11346230
4,2025-01-01T03:00:01.000Z,Checking or savings account,Checking account,Problem caused by your funds being low,Overdrafts and overdraft fees,NaN,Company believes it acted appropriately as aut...,NAVY FEDERAL CREDIT UNION,MO,65201,Servicemember,Web,2025-01-01T03:09:05.000Z,Closed with explanation,Yes,11334580


In [6]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84194 entries, 0 to 84193
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   Date received                 84194 non-null  object
 1   Product                       84194 non-null  object
 2   Sub-product                   84194 non-null  object
 3   Issue                         84194 non-null  object
 4   Sub-issue                     84143 non-null  object
 5   Consumer complaint narrative  51448 non-null  object
 6   Company public response       40070 non-null  object
 7   Company                       84194 non-null  object
 8   State                         83471 non-null  object
 9   ZIP code                      84194 non-null  object
 10  Tags                          13422 non-null  object
 11  Submitted via                 84194 non-null  object
 12  Date sent to company          84194 non-null  object
 13  Company response

In [7]:
# Step 5 — Combine the raw files safely

all_dfs = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)
    
    # Track where every record came from
    df["source_file"] = file.name
    
    all_dfs.append(df)

raw_combined = pd.concat(all_dfs, ignore_index=True)

raw_combined.shape


(566694, 17)

In [8]:
raw_combined.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID,source_file
0,2025-01-02T14:26:35.000Z,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,NaN,NaN,JPMORGAN CHASE & CO.,TN,37066,NaN,Phone,2025-01-02T14:41:19.000Z,Closed with explanation,Yes,11354262,CheckingsOrSavings.csv
1,2025-01-01T02:34:28.000Z,Checking or savings account,Checking account,Managing an account,Funds not handled or disbursed as instructed,Are deposits made to Bank of America actually ...,Company has responded to the consumer and the ...,"BANK OF AMERICA, NATIONAL ASSOCIATION",TN,37217,NaN,Web,2025-01-01T02:50:13.000Z,Closed with explanation,Yes,11345823,CheckingsOrSavings.csv
2,2025-01-01T00:54:23.000Z,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,NaN,NaN,JPMORGAN CHASE & CO.,SC,29306,Servicemember,Web,2025-01-01T01:17:26.000Z,Closed with explanation,Yes,11334911,CheckingsOrSavings.csv
3,2025-01-01T02:20:18.000Z,Checking or savings account,Savings account,Opening an account,Account opened without my consent or knowledge,NaN,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,NY,10037,NaN,Web,2025-01-01T02:53:35.000Z,Closed with explanation,Yes,11346230,CheckingsOrSavings.csv
4,2025-01-01T03:00:01.000Z,Checking or savings account,Checking account,Problem caused by your funds being low,Overdrafts and overdraft fees,NaN,Company believes it acted appropriately as aut...,NAVY FEDERAL CREDIT UNION,MO,65201,Servicemember,Web,2025-01-01T03:09:05.000Z,Closed with explanation,Yes,11334580,CheckingsOrSavings.csv


In [9]:
raw_combined.tail()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID,source_file
566689,2025-09-30T23:48:17.000Z,Debt collection,I do not know,Written notification about debt,Didn't receive enough information to verify debt,NaN,NaN,"Credit Systems International, Inc.",NC,28083,NaN,Web,2025-09-30T23:51:06.000Z,Closed with explanation,Yes,16281061,debtcollectionseptember.csv
566690,2025-06-23T19:33:13.000Z,Debt collection,Other debt,Attempts to collect debt not owed,Debt is not yours,As of XX/XX/year> the debt collection on my cr...,NaN,Resurgent Capital Services L.P.,TX,75287,NaN,Web,2025-06-23T19:43:56.000Z,Closed with explanation,Yes,14242560,debtcollectionseptember.csv
566691,2025-06-02T15:50:17.000Z,Debt collection,Federal student loan debt,Attempts to collect debt not owed,Debt is not yours,NaN,NaN,"F.H. Cann & Associates, Inc.",PA,152XX,NaN,Web,2026-08-03T18:42:48.000Z,Closed with explanation,Yes,13850464,debtcollectionseptember.csv
566692,2025-07-30T20:38:05.000Z,Debt collection,Other debt,False statements or representation,Attempted to collect wrong amount,I am submitting this complaint against Miller ...,NaN,"Peterson Young P.C., d/b/a Miller Cohen Peter...",CO,80104,NaN,Web,2025-07-30T21:16:48.000Z,Closed with explanation,No,14969012,debtcollectionseptember.csv
566693,2025-09-19T05:50:47.000Z,Debt collection,I do not know,Took or threatened to take negative or legal a...,Threatened or suggested your credit would be d...,NaN,NaN,Resurgent Capital Services L.P.,FL,32218,Servicemember,Web,2025-09-19T06:11:48.000Z,Closed with explanation,Yes,16064647,debtcollectionseptember.csv


In [10]:
# Step 6 — Check cross-file duplicates

raw_combined["Complaint ID"].nunique()

duplicate_count = raw_combined["Complaint ID"].duplicated().sum()

print("Total rows:", len(raw_combined))
print("Unique Complaint IDs:", raw_combined["Complaint ID"].nunique())
print("Duplicate Complaint IDs:", duplicate_count)

Total rows: 566694
Unique Complaint IDs: 566694
Duplicate Complaint IDs: 0


In [11]:
# Step 7 — Verify product categories

raw_combined["Product"].value_counts()

Product
Debt collection                                       283205
Credit card                                            89979
Money transfer, virtual currency, or money service     84619
Checking or savings account                            84194
Mortgage                                               24697
Name: count, dtype: int64

In [12]:
pd.crosstab(
    raw_combined["source_file"],
    raw_combined["Product"]
)

Product,Checking or savings account,Credit card,Debt collection,"Money transfer, virtual currency, or money service",Mortgage
source_file,,,,,
CheckingsOrSavings.csv,84194,0,0,0,0
CreditCard2025.csv,0,89979,0,0,0
MoneyTransfer.csv,0,0,0,84619,0
Mortgage.csv,0,0,0,0,24697
debtcollection31may.csv,0,0,81535,0,0
debtcollectiondecember.csv,0,0,80536,0,0
debtcollectionmay.csv,0,0,21959,0,0
debtcollectionseptember.csv,0,0,99175,0,0


In [13]:
# Step 8 — Verify that this is actually 2025 data

raw_combined["Date received"] = pd.to_datetime(
    raw_combined["Date received"],
    errors="coerce"
)


In [14]:
raw_combined["Date received"].min(), raw_combined["Date received"].max()

(Timestamp('2025-01-01 00:00:29+0000', tz='UTC'),
 Timestamp('2025-12-31 23:58:31+0000', tz='UTC'))

In [15]:
raw_combined["Date received"].dt.year.value_counts().sort_index()

Date received
2025    566694
Name: count, dtype: int64

In [16]:
raw_combined["Date received"].isna().sum()

np.int64(0)

In [17]:
# Step 9 — Check the debt-collection split
debt_files = raw_combined[
    raw_combined["source_file"].str.contains(
        "debtcollection",
        case=False,
        na=False
    )
]

debt_files.groupby("source_file").agg(
    rows=("Complaint ID", "size"),
    unique_ids=("Complaint ID", "nunique"),
    earliest_date=("Date received", "min"),
    latest_date=("Date received", "max")
)

,rows,unique_ids,earliest_date,latest_date
source_file,,,,
debtcollection31may.csv,81535,81535,2025-01-01 00:00:29+00:00,2025-04-30 23:58:38+00:00
debtcollectiondecember.csv,80536,80536,2025-10-01 00:02:48+00:00,2025-12-31 23:56:54+00:00
debtcollectionmay.csv,21959,21959,2025-05-01 00:01:46+00:00,2025-05-31 23:56:59+00:00
debtcollectionseptember.csv,99175,99175,2025-06-01 00:00:25+00:00,2025-09-30 23:59:46+00:00


In [18]:
# 1. Total rows / Unique Complaint IDs / Duplicate Complaint IDs

total_rows = len(raw_combined)
unique_ids = raw_combined["Complaint ID"].nunique()
duplicate_ids = raw_combined["Complaint ID"].duplicated().sum()

print("1. Complaint ID validation")
print("--------------------------")
print("Total rows:", total_rows)
print("Unique Complaint IDs:", unique_ids)
print("Duplicate Complaint IDs:", duplicate_ids)


# 2. Product value counts

print("\n2. Product counts")
print("-----------------")
print(raw_combined["Product"].value_counts())


# 3. Source file x Product crosstab

print("\n3. Source file by Product")
print("-------------------------")

product_source_crosstab = pd.crosstab(
    raw_combined["source_file"],
    raw_combined["Product"]
)

display(product_source_crosstab)


# 4. Date range and year counts

raw_combined["Date received"] = pd.to_datetime(
    raw_combined["Date received"],
    errors="coerce"
)

print("\n4. Date validation")
print("------------------")
print("Earliest date:", raw_combined["Date received"].min())
print("Latest date:", raw_combined["Date received"].max())
print("Invalid/missing dates:", raw_combined["Date received"].isna().sum())

print("\nComplaints by year:")
print(
    raw_combined["Date received"]
    .dt.year
    .value_counts()
    .sort_index()
)


# 5. Debt collection file validation

debt_files = raw_combined[
    raw_combined["source_file"]
    .str.contains("debtcollection", case=False, na=False)
]

debt_summary = (
    debt_files
    .groupby("source_file")
    .agg(
        rows=("Complaint ID", "size"),
        unique_ids=("Complaint ID", "nunique"),
        earliest_date=("Date received", "min"),
        latest_date=("Date received", "max")
    )
    .sort_values("earliest_date")
)

print("\n5. Debt Collection file summary")
print("--------------------------------")
display(debt_summary)

1. Complaint ID validation
--------------------------
Total rows: 566694
Unique Complaint IDs: 566694
Duplicate Complaint IDs: 0

2. Product counts
-----------------
Product
Debt collection                                       283205
Credit card                                            89979
Money transfer, virtual currency, or money service     84619
Checking or savings account                            84194
Mortgage                                               24697
Name: count, dtype: int64

3. Source file by Product
-------------------------


Product,Checking or savings account,Credit card,Debt collection,"Money transfer, virtual currency, or money service",Mortgage
source_file,,,,,
CheckingsOrSavings.csv,84194,0,0,0,0
CreditCard2025.csv,0,89979,0,0,0
MoneyTransfer.csv,0,0,0,84619,0
Mortgage.csv,0,0,0,0,24697
debtcollection31may.csv,0,0,81535,0,0
debtcollectiondecember.csv,0,0,80536,0,0
debtcollectionmay.csv,0,0,21959,0,0
debtcollectionseptember.csv,0,0,99175,0,0



4. Date validation
------------------
Earliest date: 2025-01-01 00:00:29+00:00
Latest date: 2025-12-31 23:58:31+00:00
Invalid/missing dates: 0

Complaints by year:
Date received
2025    566694
Name: count, dtype: int64

5. Debt Collection file summary
--------------------------------


,rows,unique_ids,earliest_date,latest_date
source_file,,,,
debtcollection31may.csv,81535,81535,2025-01-01 00:00:29+00:00,2025-04-30 23:58:38+00:00
debtcollectionmay.csv,21959,21959,2025-05-01 00:01:46+00:00,2025-05-31 23:56:59+00:00
debtcollectionseptember.csv,99175,99175,2025-06-01 00:00:25+00:00,2025-09-30 23:59:46+00:00
debtcollectiondecember.csv,80536,80536,2025-10-01 00:02:48+00:00,2025-12-31 23:56:54+00:00


In [19]:
from pathlib import Path

# Create the output folder if it doesn't already exist
output_dir = Path("../data/interim")
output_dir.mkdir(parents=True, exist_ok=True)

# Save the combined raw dataset
output_path = output_dir / "raw_combined_data.csv"
raw_combined.to_csv(output_path, index=False)

print(f"Saved successfully: {output_path}")
print(f"Saved shape: {raw_combined.shape}")

Saved successfully: ../data/interim/raw_combined_data.csv
Saved shape: (566694, 17)
